# Ejercicio de clase: Predicción del precio de vivienda con Decision Tree Regressor

**Dataset**: California Housing Prices — https://www.kaggle.com/datasets/camnugent/california-housing-prices

En el ejemplo de clase (`decision_tree_example.ipynb`) usamos un **Decision Tree Classifier** para predecir
si un paciente sobrevivía o no a una sepsis. Esa era una tarea de **clasificación**: la variable que
queríamos predecir (`hospital_outcome`) solo podía tomar un número limitado de valores (0 o 1, "vive" o
"muere").

En este ejercicio vamos a resolver un problema distinto: **regresión**. Vamos a predecir el **precio
mediano de una vivienda** (`median_house_value`) en un bloque censal de California, a partir de
características como la ubicación, la cantidad de habitaciones o el ingreso medio de sus habitantes.

> **Diferencia clave**: en clasificación el modelo predice una *categoría* (una clase). En regresión el
> modelo predice un *número real* (puede tomar, en principio, cualquier valor dentro de un rango continuo).
> Esto tiene consecuencias importantes: no podemos usar métricas como *precision*, *recall* o *F1*, porque
> esas métricas comparan clases exactas. En su lugar usaremos métricas que midan **qué tan lejos** está la
> predicción del valor real, como el **MAE (Mean Absolute Error)**, que veremos más adelante.

Para este ejercicio usarán `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.

### Antes de empezar

1. Descarguen el dataset desde Kaggle: https://www.kaggle.com/datasets/camnugent/california-housing-prices
2. El archivo se llama `housing.csv`. Colóquenlo dentro de la carpeta `raw/` de este proyecto
   (la misma carpeta `decision_tree/raw/` donde está el dataset de sepsis).
3. Sigan cada sección en orden. Cada sección tiene:
   - Una breve explicación de qué vamos a hacer y por qué.
   - Un **reto**: ustedes deben escribir el código (no está resuelto).
   - **Preguntas de análisis**: respóndanlas en una celda de markdown justo debajo de su código.


### Importar librerías

Igual que en el ejemplo de clase, empezamos importando las librerías que vamos a necesitar. Noten dos
diferencias frente al ejemplo de clasificación:

- Usamos `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.
- Usamos `mean_absolute_error` en lugar de `precision_score`, `recall_score`, `f1_score`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error


## Paso 1 — Cargar los datos

**Reto**: Carguen el archivo `housing.csv` (carpeta `raw/`) en un DataFrame de pandas llamado `df`, tal
como hicimos en el ejemplo de clase con `pd.read_csv(...)`. Luego muestren las primeras filas del
DataFrame para confirmar que se cargó correctamente.


In [ ]:
df = pd.read_csv('raw/housing.csv')

In [ ]:
df.head()

In [ ]:
df.shape

**Preguntas de análisis**
1. ¿Cuántas filas y cuántas columnas tiene el dataset? (pista: `df.shape`) R// 20640 filas y 10 columnas
2. ¿Qué representa cada fila del dataset? ¿Es una vivienda individual o algo distinto? R// Cada fila representa un bloque censal de California, no una vivienda individual.
3. Observando los nombres de las columnas, ¿cuál creen que es la variable que vamos a predecir (el *target*)? R// La variable "median_house_value"


## Paso 2 — Identificación inicial de los datos

Antes de tocar cualquier dato, siempre debemos entender con qué estamos trabajando: cuántas columnas hay,
qué tipo de dato tiene cada una, si hay valores nulos, y cuál es el rango de valores de cada variable.

**Reto**: Usando lo que ya conocen de pandas, respondan (con código) estas tres preguntas:
1. ¿Qué tipo de dato (`dtype`) tiene cada columna? ¿Hay alguna columna categórica (texto)?
2. ¿Cuáles son las estadísticas básicas (media, desviación estándar, mínimo, máximo, etc.) de las
   columnas numéricas?
3. ¿Hay valores nulos (faltantes) en el dataset? ¿En qué columna(s)?

Pistas de los métodos que necesitan (ya los usamos, en otra forma, en el ejemplo de clase): `.info()`,
`.describe()`, `.isnull()`.


In [ ]:
print("Respuesta Pregunta 1: Tipos de datos de cada columna")
df.info()

In [ ]:
print("Respuesta Pregunta 2: Estadísticas básicas de cada columna")
df.describe().T

In [ ]:
print("Respuesta Pregunta 3: Cantidad de valores nulos por columna")
df.isnull().sum()

**Preguntas de análisis**
1. ¿Cuál es la única columna categórica (de texto) del dataset? ¿Qué valores puede tomar?

   R// La columna categórica es "ocean_proximity", y puede tomar los valores: "<1H OCEAN", "INLAND", "ISLAND", "NEAR BAY", "NEAR OCEAN".

2. ¿Qué columna tiene valores nulos? ¿Cuántas filas están afectadas, aproximadamente qué porcentaje del total representa?

   R// La columna con valores nulos es "total_bedrooms", con 207 filas afectadas, lo que representa aproximadamente el 1% del total de filas (207/20640 ≈ 0.01).

3. Miren el `min` y el `max` de `median_house_value`. ¿Les parece un rango razonable para el precio de una vivienda? ¿Notan algo raro en el valor máximo? (pista: busquen cuántas filas tienen exactamente ese valor máximo).

   R// El rango de `median_house_value` es de 14999 a 500001. El valor máximo parece ser un límite impuesto, ya que hay 965 filas con exactamente ese valor máximo, lo que sugiere que podría haber un tope en los datos.

4. Comparen el rango de `median_income` con el de `total_rooms`. ¿Están en escalas muy distintas? ¿Creen que eso sería un problema para un Decision Tree? (piensen en cómo el árbol elige los cortes: ¿necesita que las variables estén en la misma escala, como sí lo necesitan otros modelos?)

   R// El rango indica que están en escalas muy distintas. Sin embargo, para un Decision Tree no es un problema que las variables estén en escalas diferentes, ya que el árbol toma decisiones basadas en la magnitud relativa de cada variable y no requiere normalización o estandarización.


In [ ]:
df['ocean_proximity'].value_counts()

In [ ]:
df['total_bedrooms'].isnull().sum() / df.shape[0]

In [ ]:
print(f"Mínimo {df['median_house_value'].min()}")
print(f"Máximo {df['median_house_value'].max()}")
print(f"Cantidad de filas con valor máximo: {df['median_house_value'].where(df['median_house_value'] == df['median_house_value'].max()).count()}")

In [ ]:
print(f"Mínimo {df['median_income'].min()}, Máximo {df['median_income'].max()}")
print(f"Mínimo {df['total_rooms'].min()}, Máximo {df['total_rooms'].max()}")

## Paso 3 — Procesamiento de datos

Por ahora, para mantener el ejercicio simple, vamos a trabajar **solo con variables numéricas**. Más
adelante en el curso aprenderemos técnicas para incorporar variables categóricas (como *one-hot encoding*),
pero hoy las vamos a descartar.

**Reto**:
1. Eliminen del DataFrame la(s) columna(s) categórica(s) que identificaron en el paso anterior.
2. Decidan qué hacer con los valores nulos que encontraron (por ejemplo, eliminar esas filas) y
   apliquen esa decisión. Un Decision Tree Regressor de scikit-learn no puede entrenarse si quedan
   valores `NaN` en los datos.
3. Confirmen, con código, que ya no quedan columnas categóricas ni valores nulos.


In [ ]:
df_num = df.drop(columns=['ocean_proximity'])

In [ ]:
df_num = df_num.dropna()

In [ ]:
df_num.info()

**Preguntas de análisis**
1. ¿Qué información de las viviendas estamos perdiendo al eliminar la columna categórica? ¿Creen que esa
   información podría ser útil para predecir el precio? ¿Por qué?
   
   R// Estamos perdiendo la informacion de la cercania al mar, creo que no es muy relevante pero podria influir si esta muy cerca ya que en general las casas cerca del mar pueden llegar a tener un buen precio.
2. Si eliminaron filas con nulos, ¿cuántas filas quedaron en total? ¿Qué porcentaje del dataset original
   se perdió?

   R// QUedaron 20433 filas, se elimiraon 207 filas las cuales representan un 1% de la informacion inicial
3. ¿Qué otra estrategia (distinta a eliminar las filas) existe para tratar valores nulos? ¿Por qué hoy
   optamos por la más simple?

   R// Existe una estrategia llamada imputacion la cual consisten en usar las medidas de tendencia central segun sea el caso para reemplazar los datos, en estte caso dado la el porcentaje de la infroamcion que se perderia en teoria, un 1% lo cual es poco significativo era posible realizar la eliminacion sin afectar de manera relevante en el resultado.

## Paso 4 — Definir variables predictoras (X) y variable objetivo (y), y dividir los datos

Igual que en el ejemplo de clase, necesitamos separar:
- **X**: las columnas que el modelo usará para predecir (todas menos el precio).
- **y**: la columna que queremos predecir (`median_house_value`).

Y luego dividir ambas en un conjunto de **entrenamiento** y uno de **prueba**, usando
`train_test_split`, tal como hicimos con los datos de sepsis.

**Reto**:
1. Construyan `X` (todas las columnas numéricas excepto `median_house_value`) y `y`
   (`median_house_value`).
2. Usen `train_test_split` para crear `X_train`, `X_test`, `y_train`, `y_test`. Usen un `test_size` de
   0.2 y `random_state=0` para que los resultados sean reproducibles.


In [ ]:
# X contiene todas las variables que usaremos para predecir
# excepto median_house_value
X = df_num.drop('median_house_value', axis=1)

# y contiene la variable que queremos predecir
y = df_num['median_house_value']

In [ ]:
# Dividimos los datos en 80% para entrenamiento y 20% para prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

**Preguntas de análisis**
1. ¿Cuántas filas quedaron en `X_train` y cuántas en `X_test`?
- X_train tiene 16346 filas y X_test tiene 4087 filas.
2. ¿Por qué es importante evaluar el modelo en datos que **no** usó para entrenar (`X_test`, `y_test`)?
   ¿Qué pasaría si evaluáramos únicamente sobre `X_train`?
- Porque queremos saber si el modelo funciona con datos nuevos. Si usamos solo X_train, puede parecer que el modelo funciona muy bien aunque solo haya aprendido esos datos.
3. En el ejemplo de clase balanceamos las clases con SMOTE antes de dividir los datos. Aquí no lo hicimos.
   ¿Por qué SMOTE (que genera ejemplos sintéticos de una *clase* minoritaria) no tiene sentido en un
   problema de regresión, donde no hay clases sino un valor continuo?
- Porque SMOTE sirve para problemas de clasificación, donde hay clases. En regresión tenemos valores numéricos continuos, como el precio de una vivienda.


## Paso 5 — Entrenar el Decision Tree Regressor

**Reto**: Entrenen un `DecisionTreeRegressor` (con `random_state=0`) usando `X_train` y `y_train`, de la
misma forma en que entrenaron el `DecisionTreeClassifier` en el ejemplo de clase.


In [ ]:
reg = DecisionTreeRegressor(random_state=0)
reg.fit(X_train, y_train)

**Preguntas de análisis**
1. En clasificación, cada hoja del árbol predice una clase (por ejemplo, "vive" o "muere"). En regresión,
   ¿qué creen que predice cada hoja del árbol? (pista: piensen en los valores de `y` que caen en esa
   hoja durante el entrenamiento).

   R// Cada hoja predice el promedio (la media) de los valores de "median_house_value" de todas las filas de entrenamiento que terminaron en esa hoja. Como "y" es un valor continuo y no una clase, el árbol no puede "votar" por una categoría; en su lugar resume el grupo con su media.

2. ¿Qué criterio usa por defecto `DecisionTreeRegressor` para decidir dónde hacer cada corte, en lugar del
   *gini* o *entropy* que se usan en clasificación? (revisen la documentación del parámetro `criterion`).

   R// Por defecto usa criterion=squared_error (error cuadrático medio, MSE). El árbol elige, en cada nodo, el corte que más reduce la varianza de y dentro de cada partición resultante, en vez de un índice de impureza de clases como *gini* o *entropy*.

## Paso 6 — Evaluar el modelo con MAE

### ¿Qué es el MAE (Mean Absolute Error)?

El **MAE** es el promedio de la diferencia absoluta entre el valor real y el valor predicho:

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

A diferencia de precision/recall/F1 (que solo tienen sentido cuando comparamos clases), el MAE funciona
sobre valores numéricos continuos y nos dice, **en promedio, cuánto se equivoca el modelo, en las mismas
unidades que la variable objetivo**. En nuestro caso, como `median_house_value` está en dólares, un
MAE de, por ejemplo, 40000 significa que en promedio el modelo se equivoca en $40,000 dólares al predecir
el precio de una vivienda.

Un MAE más bajo es mejor. Pero un mismo valor de MAE puede ser "bueno" o "malo" dependiendo de la escala
de la variable que estamos prediciendo — por eso siempre hay que compararlo contra algo (por ejemplo,
el precio promedio de las viviendas).

**Reto**:
1. Usen el modelo entrenado para predecir sobre `X_train` y calculen el MAE comparando esas predicciones
   con `y_train`.
2. Hagan lo mismo sobre `X_test` con `y_test`.
3. Comparen ambos valores.


In [ ]:
y_train_pred = reg.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
print("MAE (train):", mae_train)

In [ ]:
y_test_pred = reg.predict(X_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE (test):", mae_test)

**Preguntas de análisis**
1. ¿El MAE de entrenamiento es mayor, menor o similar al MAE de prueba? ¿Qué les dice eso sobre qué tan
   bien "memorizó" el árbol los datos de entrenamiento?

   MAE train = 0.00 y MAE test = 43,557, esto es una diferencia grandísima. Un MAE de train de 0 nos da como resultado que el árbol memorizó exactamente en cada una de las viviendas de entrenamiento. Se memorizó los datos en vez de aprenderse un patrón. El hecho de que el error se dispare tanto al pasar a datos nuevos quiere decir que esa memoria no sirve para un caso general.

2. Calculen el precio promedio (`.mean()`) de `median_house_value` en todo el dataset. Comparando ese
   promedio con el MAE de prueba, ¿el error del modelo les parece grande o pequeño en proporción al precio
   típico de una vivienda?

   El precio promedio de las viviendas en todo el dataset es $206,864. Comparado con eso, el MAE de test ($43,557) representa cerca del 21% de ese promedio, lo cual es un error bastante alto. Esto nos da que en promedio, el modelo se equivoca en más de $43,000 al predecir el precio de una vivienda, casi una cuarta parte de su valor normal.

3. En el ejemplo de clase, un árbol sin restricciones (`fully grown`) mostraba señales de sobreajuste
   (*overfitting*) al compararlo con un árbol más simple (`max_depth=3`). Según los MAE de train y test
   que obtuvieron, ¿creen que este árbol también está sobreajustado? ¿Por qué?

   Sí, está sobreajustado. El MAE train es muchísimo menor que el MAE test, y esa brecha tan grande entre ambos es justamente la señal de overfitting. Un modelo que generalizara bien tendría un MAE de test parecido al de train, mientras que aquí la diferencia es total. Finalmente, el árbol sin restricciones tuvo tanta libertad para ajustarse a los datos de entrenamiento que terminó memorizándolos en vez de aprender un patrón que se repita con datos nuevos.


## Reto adicional (opcional) — ¿Se puede mejorar el árbol?

En el ejemplo de clase, limitar la profundidad del árbol (`max_depth=3`) cambió el comportamiento del
modelo. Prueben lo mismo aquí:

1. Entrenen un segundo `DecisionTreeRegressor`, esta vez fijando `max_depth` (prueben con distintos
   valores, por ejemplo 3, 5, 10).
2. Calculen el MAE de train y de test para cada valor de `max_depth`.
3. Grafiquen (opcional) el MAE de train y de test contra `max_depth`, como una curva.

**Preguntas de análisis**
1. ¿Qué le pasa al MAE de entrenamiento a medida que aumentan `max_depth`? ¿Y al de prueba?

   MAE train: baja todo el tiempo mientras más profundo sea el árbol. A medida que tiene más profundidad se memoriza los datos y tiende a 0.

   MAE test: al principio también baja porque el árbol está aprendiendo patrones reales y útiles. Pero después de depth=10, en vez de seguir bajando, empieza a subir. Esto pasa porque, a partir de ese punto, el árbol deja de aprender patrones generales y empieza a memorizar ruido específico del set de entrenamiento, lo que lo hace ser peor.

2. ¿Existe un valor de `max_depth` donde el MAE de prueba deja de mejorar (o empeora)? ¿Qué relación tiene
   eso con el concepto de *overfitting* que vimos en el ejemplo de clase?

   Sí, el mínimo se alcanza en max_depth=10. A partir de ahí, aunque el train sigue bajando, el test empieza a subir. Esa brecha creciente entre train y test es overfitting. Una vez pasado ese punto, el árbol deja de aprender patrones generales.

3. Miren `reg.feature_importances_` del árbol entrenado. ¿Cuáles son las 2 o 3 variables más importantes
   para predecir el precio de la vivienda? ¿Tiene sentido con lo que ustedes esperarían intuitivamente?

   Las variables más importantes son median_income (0.514, muy por delante del resto), seguida de longitude (0.171) y latitude (0.160). Tiene sentido, ya que el ingreso medio de una zona determina directamente cuánto pueden pagar sus habitantes por vivienda, y la ubicación geográfica en California captura cercanía a la costa y a zonas urbanas costosas. El resto de variables (edad de la vivienda, población, dormitorios, hogares, habitaciones) aportan menos.


In [ ]:
depths = [3, 5, 10, 15, 20, 25]
results = []
for depth in depths:
    reg_d = DecisionTreeRegressor(max_depth=depth, random_state=0)
    reg_d.fit(X_train, y_train)
    mae_train_d = mean_absolute_error(y_train, reg_d.predict(X_train))
    mae_test_d = mean_absolute_error(y_test, reg_d.predict(X_test))
    results.append({'max_depth': depth, 'mae_train': mae_train_d, 'mae_test': mae_test_d})
    print(f"max_depth={depth} nos da lo siguiente: train: {mae_train_d:.2f}  test: {mae_test_d:.2f}")

results_df = pd.DataFrame(results)

etiquetas = [str(d) for d in results_df['max_depth']]
plt.plot(etiquetas, results_df['mae_train'], marker='o', label='MAE train')
plt.plot(etiquetas, results_df['mae_test'], marker='o', label='MAE test')
plt.xlabel('max_depth'); plt.ylabel('MAE'); plt.legend(); plt.show()

In [ ]:
# Importancia de cada variable en el árbol sin restricciones (reg)
importancias = pd.Series(reg.feature_importances_, index=X.columns).sort_values(ascending=False)
importancias